# Polymarket Arbitrage Opportunity Detector

## Overview
This notebook continuously scans Polymarket prediction markets to identify potential **arbitrage opportunities** - situations where purchasing positions across outcomes could theoretically yield risk-free profit regardless of the final resolution.

### Types of Arbitrage Detected
1. **Two-Outcome Markets**: When YES + NO prices sum to less than $1.00
2. **Multi-Outcome Markets**: When the sum of best asks across all outcomes < $1.00
3. **Cross-Market Arbitrage**: Mutually exclusive events across different markets

---

## Disclaimers

**IMPORTANT - READ BEFORE USE:**

1. **Educational/Informational Purpose Only**: This tool is provided for educational and research purposes. It does not constitute financial advice.

2. **No Guarantee of Profit**: Apparent arbitrage opportunities may disappear due to:
   - Price changes between detection and execution
   - Insufficient liquidity to fill orders
   - Trading fees and gas costs
   - Slippage on large orders
   - Technical errors or API delays

3. **User Responsibility**: You are solely responsible for:
   - Verifying all calculations independently
   - Understanding all risks involved
   - Complying with applicable laws and regulations
   - Any financial losses incurred

4. **Terms of Service**: Ensure your use complies with Polymarket's Terms of Service and any applicable regulations in your jurisdiction.

5. **No Warranty**: This software is provided "as is" without warranty of any kind.

---

## Cell 2: Dependencies Installation

In [ ]:
# Install required packages
!pip install -q requests pandas numpy matplotlib tabulate aiohttp nest_asyncio python-dateutil

## Cell 3: Configuration Variables

Adjust these parameters to customize the scanner behavior.

In [ ]:
# =============================================================================
# CONFIGURATION VARIABLES
# =============================================================================

# --- API Endpoints (Polymarket Public APIs) ---
BASE_URLS = {
    # Gamma API - Market metadata and discovery
    "gamma_markets": "https://gamma-api.polymarket.com/markets",
    "gamma_events": "https://gamma-api.polymarket.com/events",
    
    # CLOB API - Order book data
    "clob_host": "https://clob.polymarket.com",
    "clob_markets": "https://clob.polymarket.com/markets",
    "clob_book": "https://clob.polymarket.com/book",
    "clob_price": "https://clob.polymarket.com/price",
}

# --- Scanning Parameters ---
SCAN_INTERVAL_SECONDS = 60      # How often to re-scan markets (continuous mode)
MAX_MARKETS = 500               # Maximum number of markets to scan per cycle
MIN_LIQUIDITY_USD = 100         # Minimum market liquidity to consider (USD)
MIN_EDGE = 0.005                # Minimum profit margin to report (0.5% = 0.005)
TARGET_POSITION_SIZE = 100      # Default position size for calculations (USD)

# --- Fee Structure ---
FEE_RATE = 0.00                 # Trading fee rate (0.00 = 0%, 0.01 = 1%)
                                 # Polymarket currently has 0% maker fees
                                 # Set this if taker fees apply

# --- Slippage & Risk Parameters ---
SLIPPAGE_BUFFER = 0.005         # Additional buffer for price movement (0.5%)
MAX_SLIPPAGE_PCT = 0.02         # Maximum acceptable slippage (2%)
ORDERBOOK_DEPTH_LEVELS = 10     # Number of orderbook levels to fetch

# --- Rate Limiting ---
API_RATE_LIMIT_DELAY = 0.1      # Seconds between API calls
MAX_RETRIES = 3                 # Maximum retry attempts for failed requests
RETRY_BASE_DELAY = 1.0          # Base delay for exponential backoff (seconds)

# --- Output Options ---
SHOW_TOP_N_OPPORTUNITIES = 20   # Number of top opportunities to display
VERBOSE_LOGGING = False         # Enable detailed logging
SAVE_RESULTS_TO_CSV = True      # Save results to CSV file

# --- Authentication (Optional - for enhanced API access) ---
# Set these via Colab secrets or environment variables for authenticated endpoints
# Currently using read-only public endpoints
import os
API_KEY = os.environ.get('POLYMARKET_API_KEY', None)
API_SECRET = os.environ.get('POLYMARKET_API_SECRET', None)

print("Configuration loaded successfully!")
print(f"  - Max markets to scan: {MAX_MARKETS}")
print(f"  - Minimum edge threshold: {MIN_EDGE*100:.2f}%")
print(f"  - Fee rate: {FEE_RATE*100:.2f}%")
print(f"  - Target position size: ${TARGET_POSITION_SIZE}")

## Cell 4: Core Imports and Utilities

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import json
import logging
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from collections import defaultdict
import warnings
from tabulate import tabulate
from dateutil import parser as date_parser

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.DEBUG if VERBOSE_LOGGING else logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Display settings for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print("Core imports loaded successfully!")

## Cell 5: Data Classes and Type Definitions

In [ ]:
@dataclass
class OrderbookLevel:
    """Represents a single price level in the orderbook."""
    price: float
    size: float
    
    @property
    def value(self) -> float:
        return self.price * self.size


@dataclass
class Orderbook:
    """Represents an orderbook for a single outcome token."""
    token_id: str
    bids: List[OrderbookLevel] = field(default_factory=list)  # Buy orders (descending price)
    asks: List[OrderbookLevel] = field(default_factory=list)  # Sell orders (ascending price)
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    
    @property
    def best_bid(self) -> Optional[float]:
        return self.bids[0].price if self.bids else None
    
    @property
    def best_ask(self) -> Optional[float]:
        return self.asks[0].price if self.asks else None
    
    @property
    def spread(self) -> Optional[float]:
        if self.best_bid and self.best_ask:
            return self.best_ask - self.best_bid
        return None
    
    @property
    def total_bid_liquidity(self) -> float:
        return sum(level.value for level in self.bids)
    
    @property
    def total_ask_liquidity(self) -> float:
        return sum(level.value for level in self.asks)


@dataclass
class OutcomeToken:
    """Represents a single outcome token in a market."""
    token_id: str
    outcome: str  # e.g., "Yes", "No", "Candidate A"
    price: Optional[float] = None
    orderbook: Optional[Orderbook] = None


@dataclass
class Market:
    """Represents a Polymarket prediction market."""
    id: str
    condition_id: str
    question: str
    description: str
    outcomes: List[OutcomeToken]
    end_date: Optional[datetime] = None
    volume: float = 0.0
    liquidity: float = 0.0
    active: bool = True
    closed: bool = False
    market_slug: str = ""
    
    @property
    def is_binary(self) -> bool:
        return len(self.outcomes) == 2
    
    @property
    def num_outcomes(self) -> int:
        return len(self.outcomes)


@dataclass
class ArbitrageOpportunity:
    """Represents a detected arbitrage opportunity."""
    opportunity_type: str  # "binary", "multi_outcome", "cross_market"
    markets: List[Market]
    total_cost: float  # Cost to acquire all positions
    guaranteed_payout: float  # Guaranteed payout (usually 1.0)
    gross_profit: float  # Profit before fees
    net_profit: float  # Profit after fees
    profit_margin: float  # Net profit as percentage
    required_capital: float
    execution_plan: List[Dict[str, Any]]  # What to buy
    max_size: float  # Maximum executable size based on liquidity
    timestamp: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    confidence: str = "medium"  # low, medium, high
    
    def __str__(self) -> str:
        market_names = [m.question[:50] for m in self.markets]
        return f"Arb({self.opportunity_type}): {market_names[0]}... | Margin: {self.profit_margin*100:.2f}%"


print("Data classes defined successfully!")

## Cell 6: API Client with Retry Logic

In [ ]:
class PolymarketAPIClient:
    """
    Client for interacting with Polymarket APIs.
    Includes retry logic, rate limiting, and error handling.
    """
    
    def __init__(self, base_urls: Dict[str, str], api_key: Optional[str] = None):
        self.base_urls = base_urls
        self.api_key = api_key
        self.session = requests.Session()
        self.last_request_time = 0
        
        # Set up session headers
        self.session.headers.update({
            'Accept': 'application/json',
            'User-Agent': 'PolymarketArbitrageScanner/1.0'
        })
        
        if api_key:
            self.session.headers['Authorization'] = f'Bearer {api_key}'
    
    def _rate_limit(self):
        """Enforce rate limiting between requests."""
        elapsed = time.time() - self.last_request_time
        if elapsed < API_RATE_LIMIT_DELAY:
            time.sleep(API_RATE_LIMIT_DELAY - elapsed)
        self.last_request_time = time.time()
    
    def _request_with_retry(
        self, 
        method: str, 
        url: str, 
        params: Optional[Dict] = None,
        data: Optional[Dict] = None
    ) -> Optional[Dict]:
        """
        Make an HTTP request with exponential backoff retry logic.
        """
        self._rate_limit()
        
        for attempt in range(MAX_RETRIES):
            try:
                if method.upper() == 'GET':
                    response = self.session.get(url, params=params, timeout=30)
                elif method.upper() == 'POST':
                    response = self.session.post(url, json=data, timeout=30)
                else:
                    raise ValueError(f"Unsupported method: {method}")
                
                # Handle rate limiting
                if response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', RETRY_BASE_DELAY * (2 ** attempt)))
                    logger.warning(f"Rate limited. Waiting {retry_after}s...")
                    time.sleep(retry_after)
                    continue
                
                response.raise_for_status()
                return response.json()
                
            except requests.exceptions.RequestException as e:
                delay = RETRY_BASE_DELAY * (2 ** attempt)
                logger.warning(f"Request failed (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
                
                if attempt < MAX_RETRIES - 1:
                    logger.info(f"Retrying in {delay}s...")
                    time.sleep(delay)
                else:
                    logger.error(f"All retries exhausted for {url}")
                    return None
        
        return None
    
    def get_gamma_markets(
        self, 
        limit: int = 100, 
        offset: int = 0,
        active: bool = True,
        closed: bool = False
    ) -> List[Dict]:
        """
        Fetch markets from the Gamma API.
        """
        params = {
            'limit': limit,
            'offset': offset,
            'active': str(active).lower(),
            'closed': str(closed).lower(),
        }
        
        result = self._request_with_retry('GET', self.base_urls['gamma_markets'], params=params)
        return result if result else []
    
    def get_gamma_events(
        self,
        limit: int = 100,
        offset: int = 0,
        active: bool = True
    ) -> List[Dict]:
        """
        Fetch events (groups of related markets) from Gamma API.
        """
        params = {
            'limit': limit,
            'offset': offset,
            'active': str(active).lower(),
        }
        
        result = self._request_with_retry('GET', self.base_urls['gamma_events'], params=params)
        return result if result else []
    
    def get_clob_markets(self) -> List[Dict]:
        """
        Fetch all markets from the CLOB API.
        """
        result = self._request_with_retry('GET', self.base_urls['clob_markets'])
        return result if result else []
    
    def get_orderbook(self, token_id: str) -> Optional[Dict]:
        """
        Fetch the orderbook for a specific token.
        """
        url = f"{self.base_urls['clob_book']}"
        params = {'token_id': token_id}
        
        result = self._request_with_retry('GET', url, params=params)
        return result
    
    def get_price(self, token_id: str, side: str = 'buy') -> Optional[Dict]:
        """
        Get the current price for a token.
        """
        url = f"{self.base_urls['clob_price']}"
        params = {'token_id': token_id, 'side': side}
        
        result = self._request_with_retry('GET', url, params=params)
        return result
    
    def get_market_by_condition(self, condition_id: str) -> Optional[Dict]:
        """
        Fetch a specific market by condition ID from CLOB.
        """
        url = f"{self.base_urls['clob_markets']}/{condition_id}"
        result = self._request_with_retry('GET', url)
        return result


# Initialize the API client
api_client = PolymarketAPIClient(BASE_URLS, API_KEY)
print("API client initialized successfully!")

## Cell 7: Market Data Fetching and Parsing

In [ ]:
def parse_orderbook(orderbook_data: Dict, token_id: str) -> Orderbook:
    """
    Parse raw orderbook data into an Orderbook object.
    """
    bids = []
    asks = []
    
    if orderbook_data:
        # Parse bids (buy orders) - sorted descending by price
        for bid in orderbook_data.get('bids', []):
            try:
                price = float(bid.get('price', 0))
                size = float(bid.get('size', 0))
                if price > 0 and size > 0:
                    bids.append(OrderbookLevel(price=price, size=size))
            except (ValueError, TypeError):
                continue
        
        # Parse asks (sell orders) - sorted ascending by price
        for ask in orderbook_data.get('asks', []):
            try:
                price = float(ask.get('price', 0))
                size = float(ask.get('size', 0))
                if price > 0 and size > 0:
                    asks.append(OrderbookLevel(price=price, size=size))
            except (ValueError, TypeError):
                continue
        
        # Sort appropriately
        bids.sort(key=lambda x: x.price, reverse=True)
        asks.sort(key=lambda x: x.price)
    
    return Orderbook(token_id=token_id, bids=bids, asks=asks)


def parse_gamma_market(market_data: Dict) -> Optional[Market]:
    """
    Parse a market from Gamma API response.
    """
    try:
        market_id = market_data.get('id', '')
        condition_id = market_data.get('conditionId', market_data.get('condition_id', ''))
        question = market_data.get('question', 'Unknown')
        description = market_data.get('description', '')
        
        # Parse outcomes and token IDs
        outcomes = []
        
        # Handle different response formats
        clob_token_ids = market_data.get('clobTokenIds', [])
        outcome_names = market_data.get('outcomes', [])
        outcome_prices = market_data.get('outcomePrices', [])
        
        # Try parsing as JSON strings if needed
        if isinstance(clob_token_ids, str):
            try:
                clob_token_ids = json.loads(clob_token_ids)
            except:
                clob_token_ids = []
        
        if isinstance(outcome_names, str):
            try:
                outcome_names = json.loads(outcome_names)
            except:
                outcome_names = ['Yes', 'No']
        
        if isinstance(outcome_prices, str):
            try:
                outcome_prices = json.loads(outcome_prices)
            except:
                outcome_prices = []
        
        # Build outcome tokens
        for i, token_id in enumerate(clob_token_ids):
            outcome_name = outcome_names[i] if i < len(outcome_names) else f"Outcome {i+1}"
            price = None
            if i < len(outcome_prices):
                try:
                    price = float(outcome_prices[i])
                except (ValueError, TypeError):
                    pass
            
            outcomes.append(OutcomeToken(
                token_id=str(token_id),
                outcome=outcome_name,
                price=price
            ))
        
        # Skip markets without proper token IDs
        if not outcomes:
            return None
        
        # Parse dates and status
        end_date = None
        end_date_str = market_data.get('endDate', market_data.get('end_date_iso'))
        if end_date_str:
            try:
                end_date = date_parser.parse(end_date_str)
            except:
                pass
        
        volume = 0.0
        try:
            volume = float(market_data.get('volume', market_data.get('volumeNum', 0)) or 0)
        except:
            pass
        
        liquidity = 0.0
        try:
            liquidity = float(market_data.get('liquidity', market_data.get('liquidityNum', 0)) or 0)
        except:
            pass
        
        active = market_data.get('active', True)
        closed = market_data.get('closed', False)
        market_slug = market_data.get('slug', market_data.get('market_slug', ''))
        
        return Market(
            id=market_id,
            condition_id=condition_id,
            question=question,
            description=description,
            outcomes=outcomes,
            end_date=end_date,
            volume=volume,
            liquidity=liquidity,
            active=active,
            closed=closed,
            market_slug=market_slug
        )
    
    except Exception as e:
        logger.debug(f"Failed to parse market: {e}")
        return None


def fetch_all_markets(max_markets: int = MAX_MARKETS) -> List[Market]:
    """
    Fetch all active markets from Polymarket.
    """
    logger.info(f"Fetching up to {max_markets} active markets...")
    
    all_markets = []
    offset = 0
    batch_size = 100
    
    while len(all_markets) < max_markets:
        markets_data = api_client.get_gamma_markets(
            limit=min(batch_size, max_markets - len(all_markets)),
            offset=offset,
            active=True,
            closed=False
        )
        
        if not markets_data:
            break
        
        for market_data in markets_data:
            market = parse_gamma_market(market_data)
            if market and len(market.outcomes) >= 2:
                all_markets.append(market)
        
        if len(markets_data) < batch_size:
            break
        
        offset += batch_size
        logger.info(f"  Fetched {len(all_markets)} markets so far...")
    
    logger.info(f"Total markets fetched: {len(all_markets)}")
    return all_markets


def enrich_market_with_orderbooks(market: Market) -> Market:
    """
    Fetch orderbooks for all outcomes in a market.
    """
    for outcome in market.outcomes:
        if outcome.token_id:
            orderbook_data = api_client.get_orderbook(outcome.token_id)
            if orderbook_data:
                outcome.orderbook = parse_orderbook(orderbook_data, outcome.token_id)
                # Update price from orderbook if available
                if outcome.orderbook.best_ask:
                    outcome.price = outcome.orderbook.best_ask
    
    return market


print("Market fetching functions defined successfully!")

## Cell 8: Slippage and Cost Calculation Functions

In [ ]:
def calculate_average_fill_price(
    orderbook: Orderbook, 
    size: float, 
    side: str = 'buy'
) -> Tuple[float, float, float]:
    """
    Calculate the average fill price for a given order size.
    
    Args:
        orderbook: The orderbook to analyze
        size: Number of shares to buy/sell
        side: 'buy' (uses asks) or 'sell' (uses bids)
    
    Returns:
        Tuple of (average_price, total_cost, filled_size)
    """
    levels = orderbook.asks if side == 'buy' else orderbook.bids
    
    if not levels:
        return (0.0, 0.0, 0.0)
    
    remaining_size = size
    total_cost = 0.0
    filled_size = 0.0
    
    for level in levels:
        if remaining_size <= 0:
            break
        
        fill_at_level = min(remaining_size, level.size)
        total_cost += fill_at_level * level.price
        filled_size += fill_at_level
        remaining_size -= fill_at_level
    
    average_price = total_cost / filled_size if filled_size > 0 else 0.0
    
    return (average_price, total_cost, filled_size)


def calculate_slippage(
    orderbook: Orderbook, 
    size: float, 
    side: str = 'buy'
) -> float:
    """
    Calculate slippage as percentage for a given order size.
    
    Returns:
        Slippage as a decimal (0.01 = 1% slippage)
    """
    best_price = orderbook.best_ask if side == 'buy' else orderbook.best_bid
    
    if not best_price:
        return float('inf')
    
    avg_price, _, filled_size = calculate_average_fill_price(orderbook, size, side)
    
    if filled_size < size * 0.95:  # Less than 95% fillable
        return float('inf')
    
    if side == 'buy':
        slippage = (avg_price - best_price) / best_price if best_price > 0 else float('inf')
    else:
        slippage = (best_price - avg_price) / best_price if best_price > 0 else float('inf')
    
    return max(0, slippage)


def calculate_cost_with_fees_and_slippage(
    orderbook: Orderbook,
    size: float,
    fee_rate: float = FEE_RATE,
    slippage_buffer: float = SLIPPAGE_BUFFER
) -> Tuple[float, float]:
    """
    Calculate total cost including fees and slippage.
    
    Returns:
        Tuple of (total_cost_per_share, max_fillable_size)
    """
    avg_price, total_cost, filled_size = calculate_average_fill_price(orderbook, size, 'buy')
    
    if filled_size == 0:
        return (float('inf'), 0.0)
    
    # Add fees
    cost_with_fees = avg_price * (1 + fee_rate)
    
    # Add slippage buffer
    cost_with_buffer = cost_with_fees * (1 + slippage_buffer)
    
    return (cost_with_buffer, filled_size)


def estimate_max_arb_size(market: Market, target_margin: float = MIN_EDGE) -> float:
    """
    Estimate the maximum position size that maintains profitability.
    
    As position size increases, slippage increases and margin decreases.
    This function finds the size where margin equals target.
    """
    test_sizes = [10, 50, 100, 250, 500, 1000, 2500, 5000, 10000]
    max_profitable_size = 0
    
    for size in test_sizes:
        total_cost = 0.0
        min_fillable = float('inf')
        
        for outcome in market.outcomes:
            if outcome.orderbook:
                cost, fillable = calculate_cost_with_fees_and_slippage(
                    outcome.orderbook, size
                )
                total_cost += cost
                min_fillable = min(min_fillable, fillable)
            else:
                total_cost += outcome.price if outcome.price else 1.0
        
        margin = (1.0 - total_cost) / total_cost if total_cost > 0 else 0
        
        if margin >= target_margin and min_fillable >= size:
            max_profitable_size = size
        else:
            break
    
    return max_profitable_size


print("Slippage and cost functions defined successfully!")

## Cell 9: Arbitrage Detection Logic

In [ ]:
def detect_binary_arbitrage(
    market: Market,
    position_size: float = TARGET_POSITION_SIZE
) -> Optional[ArbitrageOpportunity]:
    """
    Detect arbitrage in a two-outcome (YES/NO) market.
    
    Arbitrage exists when: cost(YES) + cost(NO) < 1.00
    """
    if not market.is_binary:
        return None
    
    if len(market.outcomes) != 2:
        return None
    
    # Calculate costs for each outcome
    costs = []
    max_sizes = []
    execution_plan = []
    
    for outcome in market.outcomes:
        if outcome.orderbook and outcome.orderbook.best_ask:
            cost, max_size = calculate_cost_with_fees_and_slippage(
                outcome.orderbook, position_size
            )
            costs.append(cost)
            max_sizes.append(max_size)
            
            execution_plan.append({
                'action': 'BUY',
                'outcome': outcome.outcome,
                'token_id': outcome.token_id,
                'best_ask': outcome.orderbook.best_ask,
                'estimated_cost': cost,
                'available_liquidity': outcome.orderbook.total_ask_liquidity
            })
        elif outcome.price:
            # Use reported price if orderbook unavailable
            cost = outcome.price * (1 + FEE_RATE + SLIPPAGE_BUFFER)
            costs.append(cost)
            max_sizes.append(position_size)  # Unknown liquidity
            
            execution_plan.append({
                'action': 'BUY',
                'outcome': outcome.outcome,
                'token_id': outcome.token_id,
                'best_ask': outcome.price,
                'estimated_cost': cost,
                'available_liquidity': 'Unknown'
            })
        else:
            return None  # Cannot price this outcome
    
    total_cost = sum(costs)
    guaranteed_payout = 1.0
    
    # Check for arbitrage
    if total_cost >= guaranteed_payout:
        return None
    
    gross_profit = guaranteed_payout - total_cost
    # Account for fees on profit (if applicable)
    net_profit = gross_profit * (1 - FEE_RATE)
    profit_margin = net_profit / total_cost if total_cost > 0 else 0
    
    # Check minimum edge
    if profit_margin < MIN_EDGE:
        return None
    
    # Determine max executable size
    max_size = min(max_sizes) if max_sizes else 0
    
    # Confidence based on orderbook depth
    confidence = "high" if all(s >= position_size for s in max_sizes) else "medium"
    if any(s < position_size * 0.5 for s in max_sizes):
        confidence = "low"
    
    return ArbitrageOpportunity(
        opportunity_type="binary",
        markets=[market],
        total_cost=total_cost,
        guaranteed_payout=guaranteed_payout,
        gross_profit=gross_profit,
        net_profit=net_profit,
        profit_margin=profit_margin,
        required_capital=total_cost * position_size,
        execution_plan=execution_plan,
        max_size=max_size,
        confidence=confidence
    )


def detect_multi_outcome_arbitrage(
    market: Market,
    position_size: float = TARGET_POSITION_SIZE
) -> Optional[ArbitrageOpportunity]:
    """
    Detect arbitrage in a multi-outcome market.
    
    Arbitrage exists when: sum(cost(outcome_i)) < 1.00
    for all mutually exclusive outcomes.
    """
    if market.is_binary:
        return detect_binary_arbitrage(market, position_size)
    
    if len(market.outcomes) < 2:
        return None
    
    costs = []
    max_sizes = []
    execution_plan = []
    
    for outcome in market.outcomes:
        if outcome.orderbook and outcome.orderbook.best_ask:
            cost, max_size = calculate_cost_with_fees_and_slippage(
                outcome.orderbook, position_size
            )
            costs.append(cost)
            max_sizes.append(max_size)
            
            execution_plan.append({
                'action': 'BUY',
                'outcome': outcome.outcome,
                'token_id': outcome.token_id,
                'best_ask': outcome.orderbook.best_ask,
                'estimated_cost': cost,
                'available_liquidity': outcome.orderbook.total_ask_liquidity
            })
        elif outcome.price:
            cost = outcome.price * (1 + FEE_RATE + SLIPPAGE_BUFFER)
            costs.append(cost)
            max_sizes.append(position_size)
            
            execution_plan.append({
                'action': 'BUY',
                'outcome': outcome.outcome,
                'token_id': outcome.token_id,
                'best_ask': outcome.price,
                'estimated_cost': cost,
                'available_liquidity': 'Unknown'
            })
        else:
            return None
    
    total_cost = sum(costs)
    guaranteed_payout = 1.0  # One outcome must win
    
    if total_cost >= guaranteed_payout:
        return None
    
    gross_profit = guaranteed_payout - total_cost
    net_profit = gross_profit * (1 - FEE_RATE)
    profit_margin = net_profit / total_cost if total_cost > 0 else 0
    
    if profit_margin < MIN_EDGE:
        return None
    
    max_size = min(max_sizes) if max_sizes else 0
    
    confidence = "high" if all(s >= position_size for s in max_sizes) else "medium"
    if any(s < position_size * 0.5 for s in max_sizes):
        confidence = "low"
    
    return ArbitrageOpportunity(
        opportunity_type="multi_outcome",
        markets=[market],
        total_cost=total_cost,
        guaranteed_payout=guaranteed_payout,
        gross_profit=gross_profit,
        net_profit=net_profit,
        profit_margin=profit_margin,
        required_capital=total_cost * position_size,
        execution_plan=execution_plan,
        max_size=max_size,
        confidence=confidence
    )


print("Arbitrage detection functions defined successfully!")

## Cell 10: Cross-Market Arbitrage Detection

In [ ]:
# Cross-market mapping for mutually exclusive events
# Format: {"event_group": [(market_question_pattern, outcome_pattern), ...]}
CROSS_MARKET_MAPPINGS = {
    # Example: Presidential election winner markets
    "us_president_2024": [
        ("will.*win.*2024.*presidential", "Yes"),
        ("will.*be.*president", "Yes"),
    ],
}


def find_related_markets(markets: List[Market]) -> Dict[str, List[Tuple[Market, OutcomeToken]]]:
    """
    Find markets that might be related for cross-market arbitrage.
    
    Uses keyword matching and question similarity to group markets.
    """
    import re
    
    # Group markets by topic keywords
    topic_groups = defaultdict(list)
    
    # Extract keywords from questions
    keyword_patterns = [
        r'(\b\w+)\s+win',           # "X win"
        r'will\s+(\w+)\s+be',        # "will X be"
        r'(\w+)\s+vs\s+(\w+)',       # "X vs Y"
        r'(\w+)\s+election',         # "X election"
        r'(\w+)\s+president',        # "X president"
    ]
    
    for market in markets:
        question_lower = market.question.lower()
        
        # Extract key entities from the question
        entities = set()
        for pattern in keyword_patterns:
            matches = re.findall(pattern, question_lower)
            for match in matches:
                if isinstance(match, tuple):
                    entities.update(match)
                else:
                    entities.add(match)
        
        # Group by shared entities
        for entity in entities:
            if len(entity) > 3:  # Skip short words
                for outcome in market.outcomes:
                    topic_groups[entity].append((market, outcome))
    
    # Filter to only groups with multiple markets
    related_groups = {
        k: v for k, v in topic_groups.items() 
        if len(set(m.id for m, _ in v)) > 1
    }
    
    return related_groups


def detect_cross_market_arbitrage(
    markets: List[Market],
    position_size: float = TARGET_POSITION_SIZE
) -> List[ArbitrageOpportunity]:
    """
    Detect arbitrage opportunities across related markets.
    
    Looks for cases where:
    - Market A: "Will X happen?" YES price
    - Market B: "Will X NOT happen?" YES price  
    - If YES(A) + YES(B) < 1.00, arbitrage exists
    
    Or more generally:
    - Multiple markets covering mutually exclusive outcomes
    """
    opportunities = []
    related_groups = find_related_markets(markets)
    
    # Also check for explicit negation patterns
    negation_pairs = []
    
    for market in markets:
        question_lower = market.question.lower()
        
        # Look for negation pairs
        for other_market in markets:
            if market.id == other_market.id:
                continue
            
            other_lower = other_market.question.lower()
            
            # Check if one is negation of the other
            # Patterns: "will X" vs "will X not", "X wins" vs "X does not win"
            is_negation = False
            
            # Simple check: one contains "not" and otherwise similar
            if 'not' in other_lower and 'not' not in question_lower:
                # Remove 'not' and compare
                cleaned_other = other_lower.replace(' not ', ' ').replace("n't", '')
                # Check similarity (simple word overlap)
                words1 = set(question_lower.split())
                words2 = set(cleaned_other.split())
                overlap = len(words1 & words2) / max(len(words1), len(words2))
                if overlap > 0.7:
                    is_negation = True
            
            if is_negation:
                negation_pairs.append((market, other_market))
    
    # Analyze negation pairs for arbitrage
    seen_pairs = set()
    
    for market1, market2 in negation_pairs:
        pair_key = tuple(sorted([market1.id, market2.id]))
        if pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)
        
        # Get YES prices for both markets
        yes1 = None
        yes2 = None
        
        for outcome in market1.outcomes:
            if outcome.outcome.lower() == 'yes':
                if outcome.orderbook and outcome.orderbook.best_ask:
                    yes1 = outcome
                elif outcome.price:
                    yes1 = outcome
                break
        
        for outcome in market2.outcomes:
            if outcome.outcome.lower() == 'yes':
                if outcome.orderbook and outcome.orderbook.best_ask:
                    yes2 = outcome
                elif outcome.price:
                    yes2 = outcome
                break
        
        if not yes1 or not yes2:
            continue
        
        # Calculate costs
        cost1 = cost2 = 0
        max_size1 = max_size2 = position_size
        
        if yes1.orderbook and yes1.orderbook.best_ask:
            cost1, max_size1 = calculate_cost_with_fees_and_slippage(
                yes1.orderbook, position_size
            )
        elif yes1.price:
            cost1 = yes1.price * (1 + FEE_RATE + SLIPPAGE_BUFFER)
        
        if yes2.orderbook and yes2.orderbook.best_ask:
            cost2, max_size2 = calculate_cost_with_fees_and_slippage(
                yes2.orderbook, position_size
            )
        elif yes2.price:
            cost2 = yes2.price * (1 + FEE_RATE + SLIPPAGE_BUFFER)
        
        total_cost = cost1 + cost2
        
        # In a proper negation pair, exactly one YES will pay out
        guaranteed_payout = 1.0
        
        if total_cost < guaranteed_payout:
            gross_profit = guaranteed_payout - total_cost
            net_profit = gross_profit * (1 - FEE_RATE)
            profit_margin = net_profit / total_cost
            
            if profit_margin >= MIN_EDGE:
                execution_plan = [
                    {
                        'action': 'BUY',
                        'market': market1.question[:50],
                        'outcome': 'YES',
                        'token_id': yes1.token_id,
                        'estimated_cost': cost1,
                    },
                    {
                        'action': 'BUY',
                        'market': market2.question[:50],
                        'outcome': 'YES',
                        'token_id': yes2.token_id,
                        'estimated_cost': cost2,
                    }
                ]
                
                opportunities.append(ArbitrageOpportunity(
                    opportunity_type="cross_market",
                    markets=[market1, market2],
                    total_cost=total_cost,
                    guaranteed_payout=guaranteed_payout,
                    gross_profit=gross_profit,
                    net_profit=net_profit,
                    profit_margin=profit_margin,
                    required_capital=total_cost * position_size,
                    execution_plan=execution_plan,
                    max_size=min(max_size1, max_size2),
                    confidence="low"  # Cross-market arb is inherently riskier
                ))
    
    return opportunities


print("Cross-market arbitrage detection defined successfully!")

## Cell 11: Main Scanner Class

In [ ]:
class PolymarketArbitrageScanner:
    """
    Main scanner class that orchestrates the arbitrage detection process.
    """
    
    def __init__(self):
        self.markets: List[Market] = []
        self.opportunities: List[ArbitrageOpportunity] = []
        self.last_scan_time: Optional[datetime] = None
        self.scan_count = 0
    
    def fetch_markets(self, max_markets: int = MAX_MARKETS) -> None:
        """
        Fetch and store active markets.
        """
        logger.info("Fetching markets from Polymarket...")
        self.markets = fetch_all_markets(max_markets)
        logger.info(f"Fetched {len(self.markets)} markets")
    
    def enrich_with_orderbooks(self, sample_size: Optional[int] = None) -> None:
        """
        Fetch orderbooks for markets.
        Optionally limit to a sample for faster scanning.
        """
        markets_to_enrich = self.markets
        if sample_size and sample_size < len(self.markets):
            # Prioritize markets with higher liquidity
            sorted_markets = sorted(self.markets, key=lambda m: m.liquidity, reverse=True)
            markets_to_enrich = sorted_markets[:sample_size]
        
        logger.info(f"Fetching orderbooks for {len(markets_to_enrich)} markets...")
        
        for i, market in enumerate(markets_to_enrich):
            enrich_market_with_orderbooks(market)
            
            if (i + 1) % 50 == 0:
                logger.info(f"  Processed {i + 1}/{len(markets_to_enrich)} markets")
        
        logger.info("Orderbook enrichment complete")
    
    def scan_for_opportunities(self, position_size: float = TARGET_POSITION_SIZE) -> List[ArbitrageOpportunity]:
        """
        Scan all markets for arbitrage opportunities.
        """
        self.opportunities = []
        
        logger.info("Scanning for arbitrage opportunities...")
        
        # Scan binary and multi-outcome markets
        for market in self.markets:
            try:
                if market.is_binary:
                    opp = detect_binary_arbitrage(market, position_size)
                else:
                    opp = detect_multi_outcome_arbitrage(market, position_size)
                
                if opp:
                    self.opportunities.append(opp)
            except Exception as e:
                logger.debug(f"Error scanning market {market.id}: {e}")
        
        # Scan for cross-market opportunities
        try:
            cross_opps = detect_cross_market_arbitrage(self.markets, position_size)
            self.opportunities.extend(cross_opps)
        except Exception as e:
            logger.debug(f"Error in cross-market scan: {e}")
        
        # Sort by profit margin
        self.opportunities.sort(key=lambda x: x.profit_margin, reverse=True)
        
        self.last_scan_time = datetime.now(timezone.utc)
        self.scan_count += 1
        
        logger.info(f"Found {len(self.opportunities)} potential opportunities")
        
        return self.opportunities
    
    def get_opportunities_dataframe(self) -> pd.DataFrame:
        """
        Convert opportunities to a pandas DataFrame for display.
        """
        if not self.opportunities:
            return pd.DataFrame()
        
        rows = []
        for opp in self.opportunities:
            market_name = opp.markets[0].question if opp.markets else "Unknown"
            
            rows.append({
                'Type': opp.opportunity_type,
                'Market': market_name[:60] + ('...' if len(market_name) > 60 else ''),
                'Outcomes': len(opp.markets[0].outcomes) if opp.markets else 0,
                'Total Cost': f"${opp.total_cost:.4f}",
                'Payout': f"${opp.guaranteed_payout:.2f}",
                'Net Profit': f"${opp.net_profit:.4f}",
                'Margin %': f"{opp.profit_margin * 100:.2f}%",
                'Capital Req': f"${opp.required_capital:.2f}",
                'Max Size': f"${opp.max_size:.0f}",
                'Confidence': opp.confidence,
            })
        
        return pd.DataFrame(rows)
    
    def display_opportunities(self, top_n: int = SHOW_TOP_N_OPPORTUNITIES) -> None:
        """
        Display opportunities in a formatted table.
        """
        df = self.get_opportunities_dataframe()
        
        if df.empty:
            print("\n" + "="*60)
            print("No arbitrage opportunities found in current scan.")
            print("="*60)
            return
        
        print("\n" + "="*80)
        print(f"ARBITRAGE OPPORTUNITIES DETECTED - {self.last_scan_time.strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print(f"Scan #{self.scan_count} | Showing top {min(top_n, len(df))} of {len(df)} opportunities")
        print("="*80 + "\n")
        
        # Display using tabulate for nice formatting
        print(tabulate(
            df.head(top_n),
            headers='keys',
            tablefmt='grid',
            showindex=False
        ))
    
    def display_execution_plan(self, opportunity_index: int = 0) -> None:
        """
        Display detailed execution plan for an opportunity.
        """
        if not self.opportunities or opportunity_index >= len(self.opportunities):
            print("No opportunity found at that index.")
            return
        
        opp = self.opportunities[opportunity_index]
        
        print("\n" + "="*60)
        print(f"EXECUTION PLAN - Opportunity #{opportunity_index + 1}")
        print("="*60)
        print(f"Type: {opp.opportunity_type}")
        print(f"Market(s): {opp.markets[0].question}")
        print(f"Profit Margin: {opp.profit_margin * 100:.2f}%")
        print(f"Required Capital: ${opp.required_capital:.2f}")
        print(f"\nSteps to Execute:")
        print("-"*40)
        
        for i, step in enumerate(opp.execution_plan, 1):
            print(f"\nStep {i}: {step['action']} {step['outcome']}")
            print(f"  Token ID: {step['token_id']}")
            if 'best_ask' in step:
                print(f"  Best Ask: ${step['best_ask']:.4f}")
            print(f"  Est. Cost: ${step['estimated_cost']:.4f}")
            if 'available_liquidity' in step:
                liq = step['available_liquidity']
                if isinstance(liq, (int, float)):
                    print(f"  Available Liquidity: ${liq:.2f}")
                else:
                    print(f"  Available Liquidity: {liq}")
        
        print("\n" + "-"*40)
        print(f"Total Cost: ${opp.total_cost:.4f}")
        print(f"Guaranteed Payout: ${opp.guaranteed_payout:.2f}")
        print(f"Net Profit (per share): ${opp.net_profit:.4f}")
        print("="*60)
    
    def save_results(self, filename: str = None) -> str:
        """
        Save results to CSV file.
        """
        if filename is None:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f'arbitrage_opportunities_{timestamp}.csv'
        
        df = self.get_opportunities_dataframe()
        if not df.empty:
            df.to_csv(filename, index=False)
            logger.info(f"Results saved to {filename}")
        else:
            logger.info("No results to save")
        
        return filename


# Initialize the scanner
scanner = PolymarketArbitrageScanner()
print("Scanner initialized successfully!")

## Cell 12: Visualization Functions

In [ ]:
def plot_opportunity_distribution(opportunities: List[ArbitrageOpportunity]) -> None:
    """
    Create visualizations of arbitrage opportunities.
    """
    if not opportunities:
        print("No opportunities to visualize.")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Profit margin distribution
    margins = [opp.profit_margin * 100 for opp in opportunities]
    axes[0, 0].hist(margins, bins=20, color='green', edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Profit Margin (%)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Distribution of Profit Margins')
    axes[0, 0].axvline(x=MIN_EDGE*100, color='red', linestyle='--', label=f'Min Edge ({MIN_EDGE*100}%)')
    axes[0, 0].legend()
    
    # 2. Opportunity types
    type_counts = {}
    for opp in opportunities:
        type_counts[opp.opportunity_type] = type_counts.get(opp.opportunity_type, 0) + 1
    
    types = list(type_counts.keys())
    counts = list(type_counts.values())
    colors = ['#2ecc71', '#3498db', '#9b59b6']
    axes[0, 1].pie(counts, labels=types, autopct='%1.1f%%', colors=colors[:len(types)])
    axes[0, 1].set_title('Opportunities by Type')
    
    # 3. Required capital vs profit
    capitals = [opp.required_capital for opp in opportunities]
    profits = [opp.net_profit * TARGET_POSITION_SIZE for opp in opportunities]
    confidences = [opp.confidence for opp in opportunities]
    
    color_map = {'high': 'green', 'medium': 'orange', 'low': 'red'}
    colors = [color_map.get(c, 'gray') for c in confidences]
    
    axes[1, 0].scatter(capitals, profits, c=colors, alpha=0.6, s=50)
    axes[1, 0].set_xlabel('Required Capital ($)')
    axes[1, 0].set_ylabel('Expected Profit ($)')
    axes[1, 0].set_title('Capital vs Profit (color = confidence)')
    
    # Add legend for scatter plot
    for conf, color in color_map.items():
        axes[1, 0].scatter([], [], c=color, label=conf)
    axes[1, 0].legend(title='Confidence')
    
    # 4. Top opportunities bar chart
    top_n = min(10, len(opportunities))
    top_opps = opportunities[:top_n]
    labels = [f"#{i+1}" for i in range(top_n)]
    margins_top = [opp.profit_margin * 100 for opp in top_opps]
    
    bars = axes[1, 1].bar(labels, margins_top, color='teal', edgecolor='black')
    axes[1, 1].set_xlabel('Opportunity Rank')
    axes[1, 1].set_ylabel('Profit Margin (%)')
    axes[1, 1].set_title(f'Top {top_n} Opportunities by Margin')
    
    # Add value labels on bars
    for bar, margin in zip(bars, margins_top):
        axes[1, 1].text(
            bar.get_x() + bar.get_width()/2, 
            bar.get_height() + 0.1,
            f'{margin:.1f}%',
            ha='center', 
            va='bottom',
            fontsize=8
        )
    
    plt.tight_layout()
    plt.show()


def plot_market_coverage(markets: List[Market]) -> None:
    """
    Visualize market coverage statistics.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # 1. Binary vs Multi-outcome
    binary_count = sum(1 for m in markets if m.is_binary)
    multi_count = len(markets) - binary_count
    
    axes[0].pie(
        [binary_count, multi_count],
        labels=['Binary (2 outcomes)', 'Multi-outcome (3+)'],
        autopct='%1.1f%%',
        colors=['#3498db', '#e74c3c']
    )
    axes[0].set_title('Markets by Outcome Type')
    
    # 2. Liquidity distribution
    liquidities = [m.liquidity for m in markets if m.liquidity > 0]
    if liquidities:
        axes[1].hist(liquidities, bins=30, color='purple', edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('Liquidity ($)')
        axes[1].set_ylabel('Number of Markets')
        axes[1].set_title('Market Liquidity Distribution')
        axes[1].set_xscale('log')
    else:
        axes[1].text(0.5, 0.5, 'No liquidity data', ha='center', va='center')
        axes[1].set_title('Market Liquidity Distribution')
    
    # 3. Number of outcomes distribution
    outcome_counts = [len(m.outcomes) for m in markets]
    unique_counts = sorted(set(outcome_counts))
    count_freq = [outcome_counts.count(c) for c in unique_counts]
    
    axes[2].bar([str(c) for c in unique_counts], count_freq, color='teal', edgecolor='black')
    axes[2].set_xlabel('Number of Outcomes')
    axes[2].set_ylabel('Number of Markets')
    axes[2].set_title('Markets by Number of Outcomes')
    
    plt.tight_layout()
    plt.show()


print("Visualization functions defined successfully!")

## Cell 13: Run Single Scan

Execute this cell to perform a single scan of Polymarket markets.

In [ ]:
# =============================================================================
# SINGLE SCAN EXECUTION
# =============================================================================

print("Starting Polymarket Arbitrage Scan...")
print(f"Configuration: Max Markets={MAX_MARKETS}, Min Edge={MIN_EDGE*100}%")
print("="*60)

# Step 1: Fetch markets
scanner.fetch_markets(max_markets=MAX_MARKETS)

# Step 2: Enrich with orderbook data
# Note: This can be slow for many markets. Adjust sample_size to limit.
scanner.enrich_with_orderbooks(sample_size=min(200, len(scanner.markets)))

# Step 3: Scan for opportunities
opportunities = scanner.scan_for_opportunities(position_size=TARGET_POSITION_SIZE)

# Step 4: Display results
scanner.display_opportunities(top_n=SHOW_TOP_N_OPPORTUNITIES)

# Step 5: Save to CSV if enabled
if SAVE_RESULTS_TO_CSV and opportunities:
    filename = scanner.save_results()
    print(f"\nResults saved to: {filename}")

## Cell 14: View Execution Plan Details

View detailed execution plan for a specific opportunity.

In [ ]:
# View execution plan for a specific opportunity
# Change the index to view different opportunities (0 = best opportunity)

OPPORTUNITY_INDEX = 0  # Change this to view different opportunities

scanner.display_execution_plan(OPPORTUNITY_INDEX)

## Cell 15: Visualize Results

In [ ]:
# Generate visualizations of the scan results

if scanner.opportunities:
    print("Opportunity Analysis Visualizations")
    print("="*60)
    plot_opportunity_distribution(scanner.opportunities)
else:
    print("No opportunities to visualize. Run a scan first.")

if scanner.markets:
    print("\nMarket Coverage Analysis")
    print("="*60)
    plot_market_coverage(scanner.markets)

## Cell 16: Continuous Scanning Mode

Run continuous scanning with automatic refresh. **Stop execution to halt.**

In [ ]:
# =============================================================================
# CONTINUOUS SCANNING MODE
# =============================================================================
# This cell will continuously scan for opportunities.
# Press the stop button or interrupt kernel to stop.

from IPython.display import clear_output

CONTINUOUS_MODE = True  # Set to False to run only once
MAX_SCANS = 10  # Maximum number of scans (set to None for infinite)

print("Starting Continuous Arbitrage Scanner...")
print(f"Scan interval: {SCAN_INTERVAL_SECONDS} seconds")
print(f"Max scans: {MAX_SCANS if MAX_SCANS else 'Unlimited'}")
print("Press stop/interrupt to halt execution.")
print("="*60)

scan_number = 0
all_opportunities_history = []

try:
    while CONTINUOUS_MODE:
        scan_number += 1
        
        if MAX_SCANS and scan_number > MAX_SCANS:
            print(f"\nReached maximum scan limit ({MAX_SCANS}). Stopping.")
            break
        
        # Clear output for cleaner display (optional)
        # clear_output(wait=True)
        
        print(f"\n{'='*60}")
        print(f"SCAN #{scan_number} - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{'='*60}")
        
        # Fetch fresh market data
        scanner.fetch_markets(max_markets=MAX_MARKETS)
        
        # Enrich with orderbooks (sample for speed)
        scanner.enrich_with_orderbooks(sample_size=min(100, len(scanner.markets)))
        
        # Scan for opportunities
        opportunities = scanner.scan_for_opportunities(position_size=TARGET_POSITION_SIZE)
        
        # Display results
        scanner.display_opportunities(top_n=10)
        
        # Track history
        all_opportunities_history.append({
            'scan_number': scan_number,
            'timestamp': datetime.now(),
            'opportunities_count': len(opportunities),
            'best_margin': opportunities[0].profit_margin if opportunities else 0
        })
        
        # Wait before next scan
        print(f"\nNext scan in {SCAN_INTERVAL_SECONDS} seconds...")
        time.sleep(SCAN_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n\nScanning stopped by user.")

print(f"\nCompleted {scan_number} scans.")

# Summary of scanning session
if all_opportunities_history:
    history_df = pd.DataFrame(all_opportunities_history)
    print("\nScan History Summary:")
    print(history_df.to_string(index=False))

## Cell 17: Custom Market Analysis

Analyze a specific market by its condition ID or search term.

In [ ]:
def analyze_specific_market(search_term: str = None, condition_id: str = None):
    """
    Analyze a specific market in detail.
    """
    target_market = None
    
    if condition_id:
        # Search by condition ID
        for market in scanner.markets:
            if market.condition_id == condition_id:
                target_market = market
                break
    elif search_term:
        # Search by question text
        search_lower = search_term.lower()
        for market in scanner.markets:
            if search_lower in market.question.lower():
                target_market = market
                break
    
    if not target_market:
        print("Market not found. Available markets:")
        for i, m in enumerate(scanner.markets[:10]):
            print(f"  {i+1}. {m.question[:60]}...")
        return None
    
    # Ensure orderbook data is fresh
    enrich_market_with_orderbooks(target_market)
    
    print("\n" + "="*70)
    print(f"MARKET ANALYSIS")
    print("="*70)
    print(f"Question: {target_market.question}")
    print(f"Market ID: {target_market.id}")
    print(f"Condition ID: {target_market.condition_id}")
    print(f"Type: {'Binary' if target_market.is_binary else f'Multi-outcome ({len(target_market.outcomes)} outcomes)'}")
    print(f"Volume: ${target_market.volume:,.2f}")
    print(f"Liquidity: ${target_market.liquidity:,.2f}")
    
    print("\nOutcomes:")
    print("-"*50)
    
    total_cost = 0
    for outcome in target_market.outcomes:
        print(f"\n  {outcome.outcome}:")
        print(f"    Token ID: {outcome.token_id}")
        
        if outcome.orderbook:
            ob = outcome.orderbook
            print(f"    Best Bid: ${ob.best_bid:.4f}" if ob.best_bid else "    Best Bid: N/A")
            print(f"    Best Ask: ${ob.best_ask:.4f}" if ob.best_ask else "    Best Ask: N/A")
            print(f"    Spread: ${ob.spread:.4f}" if ob.spread else "    Spread: N/A")
            print(f"    Bid Liquidity: ${ob.total_bid_liquidity:.2f}")
            print(f"    Ask Liquidity: ${ob.total_ask_liquidity:.2f}")
            
            if ob.best_ask:
                total_cost += ob.best_ask
        elif outcome.price:
            print(f"    Price: ${outcome.price:.4f}")
            total_cost += outcome.price
    
    print("\n" + "-"*50)
    print(f"Total Cost to Buy All Outcomes: ${total_cost:.4f}")
    
    if total_cost < 1.0:
        profit = 1.0 - total_cost
        margin = profit / total_cost * 100
        print(f"ARBITRAGE DETECTED!")
        print(f"  Profit per share: ${profit:.4f}")
        print(f"  Margin: {margin:.2f}%")
    elif total_cost > 1.0:
        print(f"No arbitrage (cost exceeds payout by ${total_cost - 1:.4f})")
    else:
        print(f"No arbitrage (cost equals payout)")
    
    return target_market


# Example usage - search for a specific market
# Uncomment and modify to analyze a specific market:

# analyze_specific_market(search_term="president")
# analyze_specific_market(condition_id="0x...")

print("Custom market analysis function ready.")
print("Usage: analyze_specific_market(search_term='your search term')")

## Cell 18: Export Full Results

In [ ]:
def export_detailed_results(filename_prefix: str = "polymarket_arb"):
    """
    Export comprehensive results including market data and opportunities.
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export opportunities
    opp_df = scanner.get_opportunities_dataframe()
    if not opp_df.empty:
        opp_filename = f"{filename_prefix}_opportunities_{timestamp}.csv"
        opp_df.to_csv(opp_filename, index=False)
        print(f"Opportunities saved to: {opp_filename}")
    
    # Export market summary
    if scanner.markets:
        market_data = []
        for m in scanner.markets:
            # Calculate total ask cost
            total_ask = sum(
                o.orderbook.best_ask if o.orderbook and o.orderbook.best_ask else (o.price or 0)
                for o in m.outcomes
            )
            
            market_data.append({
                'Market ID': m.id,
                'Question': m.question[:100],
                'Outcomes': len(m.outcomes),
                'Volume': m.volume,
                'Liquidity': m.liquidity,
                'Total Ask Cost': total_ask,
                'Arb Potential': 'Yes' if total_ask < 1.0 else 'No',
                'Margin': f"{((1-total_ask)/total_ask)*100:.2f}%" if total_ask < 1 and total_ask > 0 else 'N/A'
            })
        
        markets_df = pd.DataFrame(market_data)
        markets_filename = f"{filename_prefix}_markets_{timestamp}.csv"
        markets_df.to_csv(markets_filename, index=False)
        print(f"Market data saved to: {markets_filename}")
    
    # Export execution plans (JSON format for complex structure)
    if scanner.opportunities:
        plans = []
        for i, opp in enumerate(scanner.opportunities):
            plans.append({
                'rank': i + 1,
                'type': opp.opportunity_type,
                'markets': [m.question for m in opp.markets],
                'total_cost': opp.total_cost,
                'net_profit': opp.net_profit,
                'margin': opp.profit_margin,
                'execution_plan': opp.execution_plan,
                'max_size': opp.max_size,
                'confidence': opp.confidence
            })
        
        plans_filename = f"{filename_prefix}_execution_plans_{timestamp}.json"
        with open(plans_filename, 'w') as f:
            json.dump(plans, f, indent=2, default=str)
        print(f"Execution plans saved to: {plans_filename}")
    
    print("\nExport complete!")


# Run export
export_detailed_results()

## Cell 19: Helper Functions and Utilities

In [ ]:
def quick_scan(max_markets: int = 50, show_top: int = 5):
    """
    Perform a quick scan with reduced market count for faster results.
    """
    print(f"Running quick scan (max {max_markets} markets)...")
    scanner.fetch_markets(max_markets=max_markets)
    scanner.enrich_with_orderbooks(sample_size=max_markets)
    scanner.scan_for_opportunities()
    scanner.display_opportunities(top_n=show_top)
    return scanner.opportunities


def filter_by_liquidity(min_liq: float = 1000):
    """
    Filter opportunities by minimum market liquidity.
    """
    filtered = [
        opp for opp in scanner.opportunities
        if all(m.liquidity >= min_liq for m in opp.markets)
    ]
    print(f"Found {len(filtered)} opportunities with liquidity >= ${min_liq}")
    return filtered


def filter_by_confidence(min_confidence: str = "medium"):
    """
    Filter opportunities by confidence level.
    """
    confidence_order = {"low": 0, "medium": 1, "high": 2}
    min_level = confidence_order.get(min_confidence, 1)
    
    filtered = [
        opp for opp in scanner.opportunities
        if confidence_order.get(opp.confidence, 0) >= min_level
    ]
    print(f"Found {len(filtered)} opportunities with confidence >= {min_confidence}")
    return filtered


def calculate_portfolio_allocation(
    opportunities: List[ArbitrageOpportunity],
    total_capital: float = 1000
) -> pd.DataFrame:
    """
    Suggest capital allocation across opportunities based on margin and confidence.
    """
    if not opportunities:
        print("No opportunities to allocate.")
        return pd.DataFrame()
    
    # Score each opportunity
    confidence_weights = {"high": 1.0, "medium": 0.6, "low": 0.3}
    
    scores = []
    for opp in opportunities:
        # Score = margin * confidence_weight * liquidity_factor
        conf_weight = confidence_weights.get(opp.confidence, 0.5)
        liq_factor = min(1.0, opp.max_size / TARGET_POSITION_SIZE) if opp.max_size > 0 else 0.5
        score = opp.profit_margin * conf_weight * liq_factor
        scores.append(score)
    
    total_score = sum(scores)
    
    if total_score == 0:
        return pd.DataFrame()
    
    # Allocate capital proportionally
    allocations = []
    for opp, score in zip(opportunities, scores):
        allocation = (score / total_score) * total_capital
        expected_profit = allocation * opp.net_profit
        
        allocations.append({
            'Market': opp.markets[0].question[:40] + '...',
            'Margin': f"{opp.profit_margin*100:.2f}%",
            'Confidence': opp.confidence,
            'Allocation': f"${allocation:.2f}",
            'Expected Profit': f"${expected_profit:.2f}"
        })
    
    df = pd.DataFrame(allocations)
    print(f"\nSuggested Capital Allocation (${total_capital} total):")
    print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
    
    return df


print("Helper functions loaded:")
print("  - quick_scan(max_markets=50, show_top=5)")
print("  - filter_by_liquidity(min_liq=1000)")
print("  - filter_by_confidence(min_confidence='medium')")
print("  - calculate_portfolio_allocation(opportunities, total_capital=1000)")

## Cell 20: Summary and Next Steps

In [ ]:
print("""
================================================================================
POLYMARKET ARBITRAGE DETECTOR - SUMMARY
================================================================================

COMPLETED FEATURES:
  [x] Market data fetching from Gamma API
  [x] Orderbook data from CLOB API
  [x] Binary market arbitrage detection (YES + NO < 1)
  [x] Multi-outcome arbitrage detection (sum of asks < 1)
  [x] Cross-market arbitrage detection (basic)
  [x] Slippage estimation from orderbook depth
  [x] Fee calculations (configurable)
  [x] Execution plans with detailed steps
  [x] Confidence scoring based on liquidity
  [x] Continuous scanning mode
  [x] Visualization and reporting
  [x] CSV/JSON export

HOW TO USE:
  1. Configure parameters in Cell 3 (MIN_EDGE, FEE_RATE, etc.)
  2. Run Cell 13 for a single scan
  3. Run Cell 14 to view execution details for opportunities
  4. Run Cell 15 for visualizations
  5. Run Cell 16 for continuous monitoring

QUICK COMMANDS:
  - quick_scan(50, 5)  # Fast scan of top 50 markets
  - filter_by_confidence('high')  # Filter to high-confidence only
  - analyze_specific_market('bitcoin')  # Deep-dive on specific market

IMPORTANT REMINDERS:
  * This is for educational purposes only
  * Always verify opportunities independently before trading
  * Markets move quickly - opportunities may disappear
  * Consider gas costs and execution risks
  * Check Polymarket Terms of Service

================================================================================
""")

# Display current state
if scanner.markets:
    print(f"Current State:")
    print(f"  - Markets loaded: {len(scanner.markets)}")
    print(f"  - Opportunities found: {len(scanner.opportunities)}")
    print(f"  - Last scan: {scanner.last_scan_time}")
else:
    print("No data loaded yet. Run Cell 13 to start scanning.")